In [ ]:
import os
import numpy as np
import xarray as xr
import zarr

import dask
import matplotlib.pyplot as plt


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_interactive_reconstructions(ds, epoches=None, variables=None):
    variables = ds.variable.values if variables is None else ds.variable.values[variables]
    epochs = ds.epoch.values if epoches is None else ds.epoch.values[epoches]
    samples = ds.sample.values

    # Pre-compute readable dates per sample (squeeze out the time_step dim)
    sample_dates = {}
    for sample in samples:
        ts = ds.timestamp.sel(sample=sample).values.squeeze().item()
        sample_dates[sample] = pd.Timestamp(ts, unit='s').strftime('%Y-%m-%d %H:%M')

    # 1. Pre-calculate scales for each variable (across all epochs/samples)
    var_scales = {}
    diff_scales = {}
    for var in variables:
        v_min = min(ds.original.sel(variable=var).min().values,
                    ds.reconstruction.sel(variable=var).min().values)
        v_max = max(ds.original.sel(variable=var).max().values,
                    ds.reconstruction.sel(variable=var).max().values)
        var_scales[var] = (v_min, v_max)
        diff = (ds.original.sel(variable=var) - ds.reconstruction.sel(variable=var)).values
        diff_abs_max = np.abs(diff).max()
        diff_scales[var] = (-diff_abs_max, diff_abs_max)

    # 2. Plotting Loop
    for epoch in epochs:
        for var in variables:
            vmin, vmax = var_scales[var]
            dmin, dmax = diff_scales[var]
            num_samples = len(samples)
            cmap_main = 'Blues' if var == 'tp' else 'magma'

            fig, axes = plt.subplots(
                num_samples, 3,
                figsize=(15, 4 * num_samples),
                constrained_layout=True
            )
            if num_samples == 1:
                axes = np.expand_dims(axes, axis=0)

            for s_idx, sample in enumerate(samples):
                orig = ds.original.sel(epoch=epoch, sample=sample, variable=var).squeeze().values
                recon = ds.reconstruction.sel(epoch=epoch, sample=sample, variable=var).squeeze().values
                orig = np.flipud(orig)
                recon = np.flipud(recon)
                diff = orig - recon
                rmse = np.sqrt(np.mean(diff ** 2))
                date_str = sample_dates[sample]

                # Per-sample diff min/max
                diff_min = diff.min()
                diff_max = diff.max()

                # Column 1: Original
                ax_orig = axes[s_idx, 0]
                im_main = ax_orig.imshow(orig, cmap=cmap_main, vmin=vmin, vmax=vmax)
                ax_orig.set_title(f"Epoch {epoch} | {var} | {date_str}\n(Original)")
                ax_orig.axis('off')

                # Column 2: Reconstruction
                ax_recon = axes[s_idx, 1]
                ax_recon.imshow(recon, cmap=cmap_main, vmin=vmin, vmax=vmax)
                ax_recon.set_title(f"Epoch {epoch} | {var} | {date_str}\n(Reconstructed)")
                ax_recon.axis('off')

                # Column 3: Difference
                ax_diff = axes[s_idx, 2]
                im_diff = ax_diff.imshow(diff, cmap='RdBu_r', vmin=dmin, vmax=dmax)
                ax_diff.set_title(
                    f"Epoch {epoch} | {var} | {date_str}\n"
                    f"(Difference  |  RMSE: {rmse:.4f})"
                )
                ax_diff.axis('off')

                # ── Annotate min/max on the difference image ──────────────────
                min_loc = np.unravel_index(np.argmin(diff), diff.shape)
                max_loc = np.unravel_index(np.argmax(diff), diff.shape)

                ax_diff.plot(min_loc[1], min_loc[0],
                             'v', color='blue', markersize=7, markeredgecolor='white',
                             markeredgewidth=0.8, label=f'Min: {diff_min:.4f}')
                ax_diff.annotate(f'Min\n{diff_min:.4f}',
                                 xy=(min_loc[1], min_loc[0]),
                                 xytext=(6, 6), textcoords='offset points',
                                 color='blue', fontsize=7, fontweight='bold',
                                 bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.6, ec='none'))

                ax_diff.plot(max_loc[1], max_loc[0],
                             '^', color='red', markersize=7, markeredgecolor='white',
                             markeredgewidth=0.8, label=f'Max: {diff_max:.4f}')
                ax_diff.annotate(f'Max\n{diff_max:.4f}',
                                 xy=(max_loc[1], max_loc[0]),
                                 xytext=(6, -14), textcoords='offset points',
                                 color='red', fontsize=7, fontweight='bold',
                                 bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.6, ec='none'))

                ax_diff.legend(loc='lower right', fontsize=7,
                               framealpha=0.7, markerscale=0.9)
                # ──────────────────────────────────────────────────────────────

            cbar_main = fig.colorbar(im_main, ax=axes[:, :2],
                                     orientation='vertical', fraction=0.02, pad=0.04)
            cbar_main.set_label(f'Value Scale for {var}')

            cbar_diff = fig.colorbar(im_diff, ax=axes[:, 2],
                                     orientation='vertical', fraction=0.04, pad=0.04)
            cbar_diff.set_label(f'Difference Scale for {var}')

            plt.show()

In [ ]:
import xarray as xr


file_paths = {
    '02-l2' : '/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/02-l2-no-norm/mse_kl_04_0_20260609_024857/samples.zarr',
    '03-lola' : '/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/03-lola-no-norm/lola_dcae_20260609_012853/samples.zarr',
    '04-qrl': '/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/04-qrl-no-norm/qrl_dcae_20260609_012522/samples.zarr',
}



In [ ]:
for label, file_path in file_paths.items():
    print(f"\n{'='*60}")
    print(f"  {label}  —  {file_path.split('/')[-2]}")
    print(f"{'='*60}")
    try:
        ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        print(ds)
        plot_interactive_reconstructions(ds, epoches=[-1], variables=[0])
    except Exception as e:
        print(f"  ERROR: {e}")

In [ ]:
for label, file_path in file_paths.items():
    print(f"\n{'='*60}")
    print(f"  {label}  —  {file_path.split('/')[-2]}")
    print(f"{'='*60}")
    try:
        ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        print(ds)
        plot_interactive_reconstructions(ds, epoches=[-1], variables=[1])
    except Exception as e:
        print(f"  ERROR: {e}")

In [ ]:
for label, file_path in file_paths.items():
    print(f"\n{'='*60}")
    print(f"  {label}  —  {file_path.split('/')[-2]}")
    print(f"{'='*60}")
    try:
        ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        print(ds)
        plot_interactive_reconstructions(ds, epoches=[-1], variables=[2])
    except Exception as e:
        print(f"  ERROR: {e}")

In [ ]:
for label, file_path in file_paths.items():
    print(f"\n{'='*60}")
    print(f"  {label}  —  {file_path.split('/')[-2]}")
    print(f"{'='*60}")
    try:
        ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        print(ds)
        plot_interactive_reconstructions(ds, epoches=[-1], variables=[3])
    except Exception as e:
        print(f"  ERROR: {e}")

In [ ]:
for label, file_path in file_paths.items():
    print(f"\n{'='*60}")
    print(f"  {label}  —  {file_path.split('/')[-2]}")
    print(f"{'='*60}")
    try:
        ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        print(ds)
        plot_interactive_reconstructions(ds, epoches=[-1], variables=[4])
    except Exception as e:
        print(f"  ERROR: {e}")

In [ ]:
for label, file_path in file_paths.items():
    print(f"\n{'='*60}")
    print(f"  {label}  —  {file_path.split('/')[-2]}")
    print(f"{'='*60}")
    try:
        ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        print(ds)
        plot_interactive_reconstructions(ds, epoches=[-1], variables=[5])
    except Exception as e:
        print(f"  ERROR: {e}")